# NCBI Field standardization exploration
NCBI does not standardize fields for genomic data, making relevant metadata difficult to track down. Here I will attempt to programmatically extract fields by their subject matter, especially pertaining to host and isolation source. 

In [195]:
import duckdb
import pandas as pd

#Create duckdb instance
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

In [ ]:
import duckdb
con = duckdb.connect()
URL = 'https://data-omicidx.cancerdatasci.org/biosample/parquet/biosamples.parquet'

#Explore OmicIDX fields
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{URL}')").show(max_width=200)
con.sql(f"SELECT num_rows, num_row_groups FROM parquet_file_metadata('{URL}')").show()


┌──────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │                                                  column_type                                                   │  null   │   key   │ default │  extra  │
│     varchar      │                                                    varchar                                                     │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ submission_date  │ VARCHAR                                                                                                        │ YES     │ NULL    │ NULL    │ NULL    │
│ last_update      │ VARCHAR                                                                                                      

In [ ]:
#Import table
con.sql(f"""
    COPY (
        SELECT accession, taxon_id, unnest(attribute_recs) AS a
        FROM read_parquet('{URL}')
        WHERE attribute_recs IS NOT NULL
    ) TO 'attrs_long.parquet' (FORMAT parquet)
""")


In [ ]:
import json
import pandas as pd

#Load bacteria-descendent Taxid list
bacteria_ids = pd.DataFrame({'taxon_id': json.load(open('../data/bacteria_ids.json'))})
con.register('bacteria_ids', bacteria_ids)

#Creaate filtered bacteria-only table
con.sql("""
    COPY (
        SELECT l.*
        FROM 'attrs_long.parquet' l
        SEMI JOIN bacteria_ids b USING (taxon_id)
    ) TO 'attrs_long_bacteria.parquet' (FORMAT parquet)
""")


In [ ]:
#Create a table of keys with harmonized name, number of samples represented and the unique field values
catalog = con.sql("""
    SELECT lower(trim(a.attribute_name)) AS key,
           any_value(a.harmonized_name)  AS harmonized,   -- drop if not in schema
           count(*)                      AS n_rows,
           count(DISTINCT accession)     AS n_samples,
           count(DISTINCT a.value)       AS n_distinct_vals,
           list(DISTINCT a.value)[1:5]   AS example_vals
    FROM 'attrs_long_bacteria.parquet'
    GROUP BY 1
    ORDER BY n_samples DESC
""").df()


In [165]:
catalog.head()

,key,harmonized,n_rows,n_samples,n_distinct_vals,example_vals
0,isolation_source,isolation_source,4871799,4871095,82606,"[water, neonate gastric aspirate, iceberg, Hea..."
1,collection_date,collection_date,3756174,3756174,22789,"[2012-10, 2008-07-11, 2010-11, 2013, 2005-01-20]"
2,geo_loc_name,geo_loc_name,3722754,3722754,36771,"[Ecuador, Germany: Bavaria, USA: Connecticut, ..."
3,sample name,sample_name,3067246,3067246,2956376,"[BROAD:SEQUENCING_SAMPLE:24276.0, BROAD:SEQUEN..."
4,insdc center name,NaN,3035967,3035967,2552,"[FORSCHUNGSZENTRUM BORSTEL, Department of Food..."


In [196]:
catalog = pd.read_csv(r"../data/catalog.csv")

Examining host related columns

In [197]:
#Identifying NCBI host-mapped field names
host_mask = catalog['harmonized'] == 'host'
catalog[host_mask]['key'].to_list()

['host',
 'host scientific name',
 'specific host',
 'specific_host',
 'nat-host',
 'nat_host',
 'host_species',
 'host species',
 'host name',
 'specific-host',
 'host organism']

In [198]:
uniq_host_vals = catalog[catalog['harmonized'] == 'host']['n_distinct_vals'].sum()
print(f"{uniq_host_vals} unique host values captured by NCBI mapping")

19477 unique host values captured by NCBI mapping


Identifying other host-related harmonized fields

In [199]:
harmonized_host_cols = catalog[catalog['harmonized'].str.contains("host", case=False)]['harmonized'].unique()
#Convert to list
harmonized_host_cols = list(harmonized_host_cols)
harmonized_host_cols

['host',
 'host_disease',
 'host_health_state',
 'host_age',
 'host_sex',
 'host_tissue_sampled',
 'host_subject_id',
 'host_description',
 'host_disease_outcome',
 'host_disease_stage',
 'host_taxid',
 'host_life_stage',
 'host_common_name',
 'lab_host',
 'host_body_product',
 'host_body_habitat',
 'host_diet',
 'host_genotype',
 'host_variety',
 'host_animal_breed',
 'host_housing',
 'host_group_size',
 'host_am',
 'host_hiv_stat',
 'host_phenotype',
 'host_tot_mass',
 'host_height',
 'host_length',
 'host_dependence',
 'host_infra_specific_name',
 'host_infra_specific_rank',
 'host_dry_mass',
 'host_wet_mass',
 'host_body_temp',
 'host_body_mass_index',
 'host_family_relationship',
 'host_last_meal',
 'host_occupation',
 'host_substrate',
 'host_pulse',
 'host_subspecf_genlin',
 'host_color',
 'host_growth_cond',
 'host_shape',
 'host_blood_press_syst',
 'host_blood_press_diast',
 'host_number',
 'host_symbiont',
 'host_cellular_loc',
 'host_specificity',
 'host_specimen_voucher',
 

Harmonized fields which contain the string 'host' vary a lot - I will manually seperate them into fields that identify the host and fields with related host information

In [200]:
#seperate host identifying harmonizers from host-related harmonizers
harmonized_host_identifier = ['host_common_name', 'host_taxid', 'host', 'host_infra_specific_name', 'host_infra_specific_rank', 'host_animal_breed', 'host_variety', 'host_subspecf_genlin', 'host_of_host_sub_id', 'host_of_host_infrank', 'host_of_host_name', 'host_of_host_taxid', 'host_of_host_infname']
harmonized_host_context = list(set(harmonized_host_cols)- set(harmonized_host_identifier))

Now I will identify keys with the word 'host' but without a host harmonizer

In [201]:
#Mask for all positive host harmonizers
all_host_harmonizers = ((catalog['harmonized'].isin(harmonized_host_identifier))|(catalog['harmonized'].isin(harmonized_host_context)))

#Exclude all host harmonizers
not_host_harmonized = catalog[~all_host_harmonizers]

#All remaining keys with 'host'
not_host_harmonized_host_keys = not_host_harmonized['key'].str.contains('host', case=False)
not_host_harmonized_host_keys = not_host_harmonized[not_host_harmonized_host_keys]


print(len(not_host_harmonized_host_keys['key'].to_list()))
print(not_host_harmonized_host_keys[['key', 'harmonized', 'n_samples']].sort_values(by='n_samples', ascending=False))

129
                                             key harmonized  n_samples
206   is the sequenced pathogen host associated?        NaN      16196
244             isolation source host associated        NaN      11686
257                              host associated        NaN      10864
284                              host_associated        NaN       8147
403             isolation source host-associated        NaN       4333
...                                          ...        ...        ...
7133                                  host state        NaN          1
7253                                       ihost        NaN          1
7325                                habitat host        NaN          1
7500                          dhost_isease_stage        NaN          1
7545                       host_isolation_source        NaN          1

[129 rows x 3 columns]


There are 129 field values which contain the word 'host' but are not mapped to a host harmonizer. I'll map them to identifying host info, and host context info

In [202]:
df = not_host_harmonized_host_keys
print(len(df))
df.to_csv(r"../data/host_keys.csv")

129


I used the exported csv to manually identify fields that contain identifying information for the host

In [203]:
host_identifier_keys_unmapped = ['host-associated environmental package', 'host breed', 'host_group', 'host taxon name', 'host type', 'host_information', 'host(click_for_ncbi_taxo_brower)', 'host_breed', "'host'", 'mammal host',
                                  'host plant', 'host_of_insect', 'organism host', 'full_host_name,', 'ihost']

host_context_keys_unmapped = list(set(df['key'].to_list())-set(host_identifier_keys_unmapped))

Lastly, some of the host-related fields are too unstructured to maintain as metadata and should be removed

In [204]:
host_removal = ['host_isolate', 'voucherofhost', 'host_sra_readset',  'host culture id', 'host_rootstock', 'host_fev1_at_time_of_collection', 
                'host npgs accession', 'date host collection', 'experiment host strain', 'host biosample', 'host_age_in_months', 'host bs']

print(f"Before removing unwanted host fields: {len(host_context_keys_unmapped)}")
host_context_keys_unmapped = list(set(host_context_keys_unmapped)-set(host_removal))
print(f"After removing unwanted host fields: {len(host_context_keys_unmapped)}")

Before removing unwanted host fields: 116
After removing unwanted host fields: 104


Now we'll add a mapping column to the catalog, assigning host_name and host_comments respectively. We can use this column for more mapping later

In [216]:
import numpy as np

# Each category matches if the raw key OR its harmonized name is in the curated lists
is_identifier = (catalog['key'].isin(host_identifier_keys_unmapped) |
                 catalog['harmonized'].isin(harmonized_host_identifier))
is_context    = (catalog['key'].isin(host_context_keys_unmapped) |
                 catalog['harmonized'].isin(harmonized_host_context))

# Catch keys curated into both categories before assigning
conflicts = catalog[is_identifier & is_context]
print(f"{len(conflicts)} keys match both categories")
display(conflicts[['key', 'harmonized']])

catalog['manual_map'] = np.select(
    [is_identifier, is_context],
    ['host_name', 'host_comments'],
    default=None,
)

'''
harmonized_host_context
host_context_keys_unmapped
harmonized_host_identifier
host_identifier_keys_unmapped'''

0 keys match both categories


,key,harmonized


'\nharmonized_host_context\nhost_context_keys_unmapped\nharmonized_host_identifier\nhost_identifier_keys_unmapped'

In [217]:
catalog.groupby('manual_map')['n_samples'].sum()

manual_map
host_comments    2971864
host_name        3090660
Name: n_samples, dtype: int64

### Isolation source field identification   
Just as we did with host, we'll now identify the harmonizers and keys related to isolation source

In [207]:
catalog[['harmonized', 'n_samples']].groupby(by= 'harmonized')['n_samples'].sum().sort_values(ascending=False)

harmonized
geo_loc_name                 6241915
collection_date              6216605
isolation_source             5515713
sample_name                  3333604
host                         3027516
                              ...   
association_duration               1
urogenit_disord                    1
animal_diet                        1
animal_feed_equip                  1
antiviral_treatment_agent          1
Name: n_samples, Length: 638, dtype: int64

In [ ]:
#Identifying NCBI isolation-mapped field names

isolation_related_harmonized = ['source_name', 'collected_by', 'env_broad_scale', 'env_medium', 'env_local_scale',  'isolation_source']
harmonizer_iso_mask = catalog['harmonized'].isin(isolation_related_harmonized)

#List of keys included by harmonized isolation source fields
iso_keys = catalog[harmonizer_iso_mask]['key'].to_list()
print(len(iso_keys))
print(iso_keys)

50
['isolation_source', 'collected_by', 'environmental medium', 'broad-scale environmental context', 'env_broad_scale', 'env_medium', 'env_local_scale', 'isolation source', 'environment (feature)', 'environment (biome)', 'environment (material)', 'env_biome', 'source_name', 'isolation-source', 'collected by', 'env_feature', 'env_material', 'biome', 'material', 'feature', 'source name', 'collected-by', 'isolate_source', 'local-scale environmental context', 'environment_biome', 'broad scale environmental context', 'isolate source', 'source of isolate', 'environment feature', 'local scale environmental context', 'environmental_medium', 'environment biome', 'broad-scale_environmental_context', 'environment material', 'local-scale_environmental_context', 'env_local scale', 'env material', 'env medium', 'env local scale', 'env broad scale', 'environment_material', 'local_scale_environmental_context', 'isolation   source', 'environment_feature', 'broad_scale_environmental_context', 'collected

In [209]:
import re

isolation_synonyms = ['isolat', 'source', 'sample_type', 'env_', 'environment', 'body_site', 'tissue', 'material', 'habitat', 'origin']
synonym_pattern = '|'.join(re.escape(s) for s in isolation_synonyms)

#Regex search of isolation synonyms
key_iso_mask = (~catalog['key'].isin(iso_keys)) & (catalog['key'].str.contains(synonym_pattern, case=False, na=False))
print(sum(key_iso_mask))

iso_catalog = catalog[key_iso_mask]

#Export the isolation source key candidates for manual review
iso_catalog[['key','n_samples', 'n_distinct_vals','example_vals']].sort_values(by='n_samples', ascending=False).to_csv(r"../data/isolation_keys.csv")




480


I manually reviewed the keys returned by key_iso_mask to see if they contain relevant information. The general exclusions were:
* Values of the field do not require external information to make sense of (ie only ontology codes, study-specific identifiers)
* Dates, coordinates

In [210]:
iso_candidates_reviewed = pd.read_csv(r"../data/isolation_keys_reviewed.csv")

#false values are valid keys for isolation source. Add the validated keys 
final_iso_list = iso_keys + iso_candidates_reviewed[iso_candidates_reviewed['not_valid_iso_field'] == False]['key'].to_list()



In [ ]:
key_iso_mask = catalog['key'].isin(final_iso_list)

#Create mapping column 
catalog['manual_map'] = np.select(
    [is_identifier, is_context, key_iso_mask],
    ['host_name', 'host_comments', 'isolation_comments'],
    default=None,
)

In [222]:
catalog.groupby('manual_map')['n_distinct_vals'].sum()

manual_map
host_comments           82532
host_name               21412
isolation_comments    1446471
Name: n_distinct_vals, dtype: int64

Saving the mapping table to csv

In [ ]:
key_map = catalog[['key', 'manual_map']].dropna(how='any')



391

In [229]:
key_map.to_csv(r"../data/key_map.csv")

In [ ]:
catalog.to_csv("../data/catalog.csv")